# tocandira: label every position at depths 3, 4 and 5

Searches the positions of games already on Drive again, as a function of the position only:
for every position and depth, a fresh engine (one thread, 16 MiB table, root order seeded
by the position) searches it. The same position gets the same score in any game, order or
session. Output per shard: the relabeled games (`data/m/<source>-shard-NNN.jsonl.gz`) and
their radius 6 encoding (`data/mr6/<source>-shard-NNN.npz`, layout version 3 with `score3`,
`score4`, `score5` and `opening`), so 02_train reads it with `DATASET = "m"` and encodes
nothing.

One notebook file per parallel run: save a copy in Drive per instance, set `INSTANCE` to 0
in one copy and 1 in the other. Instances take disjoint shards; a finished shard (its
`.jsonl.gz` exists) is skipped, so a disconnected session just gets re-run. The depth 4
games come first (night 1, ~8 h on v6e-1 + v5e-1), then the depth 5 games (night 2, ~5 h).

CPU bound: pick the runtime with the most CPUs (v6e-1: 44, v5e-1: 24).

In [ ]:
INSTANCE, N_INSTANCES = 0, 2   # this copy does tasks with index % N_INSTANCES == INSTANCE
SOURCES = [("d4", 4), ("d5", 5)]  # data/<folder> with shard-NNN.jsonl.gz, depth the games were played at
SHARDS = 30
DEPTHS = "3,4,5"
DATA = "/content/drive/MyDrive/tocandira/data"

import os
print("cpus:", os.cpu_count())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DATA}/m", exist_ok=True)
os.makedirs(f"{DATA}/mr6", exist_ok=True)

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then curl -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null; fi
source "$HOME/.cargo/env"
cd /content
# Cloned to /content/repo, not /content/tocandira: a folder named like the package would shadow it on sys.path.
if [ -d repo ]; then (cd repo && git pull -q && git submodule update --init -q); else git clone -q --recurse-submodules https://github.com/abstratovcm/tocandira repo; fi
cd /content/repo/engine/nokamute
CARGO_PROFILE_RELEASE_LTO=true CARGO_PROFILE_RELEASE_CODEGEN_UNITS=1 cargo build --release -q
# Not an editable install: its .pth file would only be read when the kernel restarts.
cd /content/repo && pip install -q .
./engine/nokamute/target/release/nokamute --help | head -1

In [ ]:
import gzip
import shutil
import subprocess
import time

WORKERS = os.cpu_count() or 2
os.environ["TOCANDIRA_ENGINE"] = "/content/repo/engine/nokamute/target/release/nokamute"
tasks = [(folder, depth, shard) for folder, depth in SOURCES for shard in range(SHARDS)]
for folder, depth, shard in tasks[INSTANCE::N_INSTANCES]:
    name = f"{folder}-shard-{shard:03d}"
    npz, gz = f"{DATA}/mr6/{name}.npz", f"{DATA}/m/{name}.jsonl.gz"
    if os.path.exists(gz):
        print("exists, skipping", gz)
        continue
    t0 = time.time()
    jsonl = f"/content/{name}.jsonl"
    subprocess.run(
        ["hive-label", "--relabel", f"{DATA}/{folder}/shard-{shard:03d}.jsonl.gz",
         "--depths", DEPTHS, "--source-depth", str(depth), "--workers", str(WORKERS),
         "--out", jsonl],
        check=True,
    )
    subprocess.run(["hive-encode", jsonl, "--radius", "6", "--out", "/content/shard.npz",
                    "--workers", str(WORKERS)], check=True)
    # The npz first and the move lists last, each under a temporary name: 02_train only
    # sees finished shards (it globs the .jsonl.gz) and never encodes one itself.
    shutil.copyfile("/content/shard.npz", npz + ".part")
    os.replace(npz + ".part", npz)
    with open(jsonl, "rb") as src, gzip.open(gz + ".part", "wb") as dst:
        shutil.copyfileobj(src, dst)
    os.replace(gz + ".part", gz)
    os.remove(jsonl)
    os.remove("/content/shard.npz")
    print(f"{name}: {time.time() - t0:.0f}s -> {gz}")

Reference: 14 workers on a Ryzen 7 5700G label about 16 positions/s at depths 3+4+5 (depth
5 is nearly all of it), about 50/s on v6e-1: a depth 4 shard (77k positions) takes about 26
minutes there, a depth 5 shard (49k) about 16; v5e-1 about half that speed. Two runs of the
same shard give identical files.